# 00 — Setup y bootstrap del proyecto P10JJ

Este notebook **se ejecuta antes que el resto**. Su trabajo es dejar el entorno listo,
tanto si lo abres en tu máquina local como en **Colab**, **Kaggle** o **Codespaces**.

Cuando termines de ejecutarlo verás un resumen del entorno detectado y de qué claves de API están cargadas.


## 1. Detectar entorno


In [ ]:
import os
import platform
import subprocess
import sys
from contextlib import suppress


def en_colab():
    return "google.colab" in sys.modules


def en_kaggle():
    return os.path.exists("/kaggle")


def en_codespaces():
    return os.environ.get("CODESPACES") == "true"


ENTORNO = (
    "colab"
    if en_colab()
    else "kaggle"
    if en_kaggle()
    else "codespaces"
    if en_codespaces()
    else "local"
)
print(f"Entorno detectado: {ENTORNO}")
print(f"Python:            {sys.version.split()[0]}")
print(f"Plataforma:        {platform.platform()}")

## 2. Instalar dependencias mínimas si faltan

En **local** y **Codespaces** asumimos que ya hiciste `uv sync` y todo está instalado.
En **Colab/Kaggle** instalamos con `pip` lo justo para que el notebook funcione.


In [ ]:
REQ_MIN = [
    "langchain>=0.3,<0.4",
    "langchain-groq",
    "langchain-google-genai",
    "langchain-community",
    "chromadb>=0.5,<0.6",
    "sentence-transformers",
    "python-dotenv",
    "streamlit",
]


def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])


if ENTORNO in ("colab", "kaggle"):
    print("Instalando dependencias mínimas...")
    pip_install(REQ_MIN)
    print("OK.")
else:
    print("Entorno local/Codespaces: se asume que `uv sync` ya está hecho.")

## 3. Cargar claves de API

- **Local / Codespaces**: leemos `.env` desde la raíz del repo.
- **Colab**: leemos de Secrets (panel de la izquierda, icono de llave).
- **Kaggle**: leemos de Secrets (Add-ons → Secrets).

Las claves esperadas son: `GROQ_API_KEY`, `GOOGLE_API_KEY`, `LANGSMITH_API_KEY`, `HUGGINGFACEHUB_API_TOKEN`.


In [ ]:
CLAVES = ["GROQ_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY", "HUGGINGFACEHUB_API_TOKEN"]

if ENTORNO == "colab":
    try:
        from google.colab import userdata

        for k in CLAVES:
            with suppress(Exception):
                os.environ[k] = userdata.get(k)
    except Exception as e:
        print(f"Aviso: no se pudo acceder a Colab userdata: {e}")

elif ENTORNO == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for k in CLAVES:
            with suppress(Exception):
                os.environ[k] = client.get_secret(k)
    except Exception as e:
        print(f"Aviso: no se pudo acceder a Kaggle Secrets: {e}")

else:
    # local / codespaces
    from dotenv import load_dotenv

    load_dotenv()

# Trazabilidad LangSmith (Nivel Avanzado): solo se activa si hay clave.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", "P10JJ")

print("Claves cargadas:")
for k in CLAVES:
    estado = "OK" if os.getenv(k) else "FALTA"
    print(f"  {k:30s} {estado}")

## 4. Prueba rápida — Groq

Si tienes `GROQ_API_KEY`, esta celda hace una llamada mínima al modelo y confirma que todo está bien conectado.
Si falla por falta de clave, no pasa nada: rellena `.env` y reejecuta esta celda.


In [ ]:
if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    resp = llm.invoke("Saluda en una frase corta, en español.")
    print(resp.content)
else:
    print("GROQ_API_KEY no configurada. Pega tu clave en .env y reejecuta esta celda.")

## 5. (Opcional) Importar nuestro paquete `p10jj`

Solo aplica en local/Codespaces donde el paquete está instalado con `uv sync`.
En Colab/Kaggle aún no tiene sentido (el paquete vive solo en tu repo).


In [ ]:
if ENTORNO in ("local", "codespaces"):
    try:
        import p10jj

        print(f"p10jj importado correctamente. Versión: {getattr(p10jj, '__version__', 'n/d')}")
    except ImportError:
        print("p10jj aún no instalado. Lanza `uv sync` desde la raíz del repo.")
else:
    print(f"Saltado en entorno '{ENTORNO}'.")

---

**Listo.** Si las claves están cargadas y la prueba de Groq imprime una respuesta, el entorno está preparado.

Siguiente notebook recomendado: `01_esencial_prompts.ipynb`.
